# 03 - Video demo

Runs the trained detector plus the table tracker over a video and produces:

* `annotated.mp4` - every table with a stable ID, its state and how long it has been in it
* `events.json` - state changes and dirty-table alerts, in video time
* `summary.json` - per-table time in each state, dirty episodes, mean time until cleaned

In [ ]:
# This project's repo
REPO_URL = "https://github.com/kvarun10/Smart-Restaurant-Monitoring-System.git"

import os, sys
REPO = "/content/tablewatch"
if not os.path.isdir(REPO):
    !git clone -q {REPO_URL} {REPO}
else:
    !git -C {REPO} pull -q
if f"{REPO}/src" not in sys.path:
    sys.path.insert(0, f"{REPO}/src")
!pip -q install roboflow ultralytics
print("code version:", os.popen(f"git -C {REPO} log -1 --format=%h_%s").read().strip())

In [ ]:
from google.colab import files
print("Upload the weights (.pt) from notebook 02, then a test video (.mp4)")
up = files.upload()
WEIGHTS = next(f for f in up if f.endswith(".pt"))
VIDEO = next(f for f in up if not f.endswith(".pt"))

In [ ]:
import json
from tablewatch.pipeline import run_video
from tablewatch.tracker import TrackerConfig

OUT = "/content/report/03_video_demo"
# dirty_alert_s is 5 minutes in a real restaurant; a short demo clip needs a short threshold.
cfg = TrackerConfig(dirty_alert_s=20)
summary = run_video(WEIGHTS, VIDEO, OUT, cfg, conf=0.35, stride=2)
json.dump({"weights": WEIGHTS, "video": VIDEO, "conf": 0.35, "stride": 2, "tracker": cfg.__dict__},
          open(f"{OUT}/settings.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
# Re-encode to H.264 so the browser can play it, then show it.
!ffmpeg -y -loglevel error -i {OUT}/annotated.mp4 -vcodec libx264 -pix_fmt yuv420p {OUT}/annotated_h264.mp4
from IPython.display import Video
Video(f"{OUT}/annotated_h264.mp4", embed=True, width=800)

In [ ]:
import pandas as pd
events = pd.DataFrame(json.load(open(f"{OUT}/events.json")))
events

In [ ]:
import shutil
shutil.make_archive("/content/03_video_demo", "zip", OUT)
files.download("/content/03_video_demo.zip")